# CSP-4-Scheduling-Csharp : Problèmes d'Ordonnancement avec Choco-solver via IKVM

**Navigation** : [<< CSP-3-Advanced](CSP-3-Advanced.ipynb) | [Index](../README.md) | [CSP-5-Optimization >>](CSP-5-Optimization.ipynb)

> **Durée estimée** : ~1h30 | **Prérequis** : [CSP-3-Advanced](CSP-3-Advanced.ipynb), [Sudoku-0-Environment-Csharp](../../Sudoku/Sudoku-0-Environment-Csharp.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. **Modéliser** un problème d'ordonnancement industriel (JSSP, RCPSP, Nurse) avec Choco-solver
2. **Utiliser** les contraintes globales Choco (`noOverlap`, `cumulative`, `allDifferent`) depuis C#/.NET
3. **Comparer** l'API Java de Choco avec l'API Python d'OR-Tools CP-SAT sur un même problème
4. **Exploiter** le bridge IKVM 8.15.0 + DLL Choco pré-compilée pour exécuter du Java dans le kernel .NET Interactive

## Pourquoi ce notebook ?

Le notebook [CSP-4-Scheduling.ipynb](CSP-4-Scheduling.ipynb) (version Python) présente le JSSP, le RCPSP et le Nurse Scheduling avec OR-Tools CP-SAT. **Ce binôme .NET** reprend les trois mêmes problèmes en utilisant **Choco-solver**, une librairie Java de CP portée via IKVM. Objectif : démontrer la **parité Python/.NET** de la série CSP (cf. epic [#4956](https://github.com/jsboige/CoursIA/issues/4956)) en réutilisant la jurisprudence [Sudoku-11-Choco-Csharp](../../Sudoku/Sudoku-11-Choco-Csharp.ipynb) (IKVM 8.15.0 + `org.chocosolver.solver.dll`).

> *Ancres savantes -- Brucker, P. (2007), Scheduling Algorithms (5e éd.), Springer (job-shop scheduling et ordonnancements à contraintes de ressources) ; Herroelen, W., De Reyck, B. & Demeulemeester, E. (1998), Resource-Constrained Project Scheduling: A Survey of Recent Developments, Computers & Operations Research 25(4):279-302 (RCPSP, méthodes exactes et heuristiques) ; Burke, E.K. et al. (2004), The State of the Art of Nurse Rostering, Journal of Scheduling 7(6):441-499 (survey de référence sur le Nurse Scheduling Problem).*

In [1]:
// Configuration du répertoire de travail
// Recherche le répertoire Part2-CSP depuis le cwd, ou remonte dans l'arborescence
using System;
using System.IO;

string FindCspDir()
{
    var dir = new DirectoryInfo(Directory.GetCurrentDirectory());
    while (dir != null)
    {
        // Check if we're in Part2-CSP directory
        if (File.Exists(Path.Combine(dir.FullName, "CSP-1-Fundamentals.ipynb")))
            return dir.FullName;
        // Check if Part2-CSP is a subdirectory
        var candidate = Path.Combine(dir.FullName, "MyIA.AI.Notebooks", "Search", "Part2-CSP");
        if (Directory.Exists(candidate) && File.Exists(Path.Combine(candidate, "CSP-1-Fundamentals.ipynb")))
            return candidate;
        dir = dir.Parent;
    }
    return Directory.GetCurrentDirectory();
}

var cspDir = FindCspDir();
Directory.SetCurrentDirectory(cspDir);
Console.WriteLine($"Répertoire de travail: {Path.GetFileName(cspDir.TrimEnd(Path.DirectorySeparatorChar, Path.AltDirectorySeparatorChar))}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

## Configuration IKVM et Choco-solver

### Approche : DLL pré-compilée + runtime IKVM NuGet

Le JAR `choco-solver-4.10.17` a été pré-compilé en DLL .NET avec IKVM 8.15.0 :
- **`org.chocosolver.solver.dll`** (12 Mo) : Choco-solver compilé, situé à côté de ce notebook

Le runtime IKVM lui-même est chargé via NuGet (`IKVM`, `IKVM.Image`, `IKVM.Image.runtime.win-x64` en 8.15.0), ce qui évite d'épingler une version précise de `System.Text.Json`.

### Compilation de la DLL Choco

La DLL Choco a été produite via un projet .NET temporaire utilisant `<IkvmReference>` :

```xml
<!-- ChocoIkvm.csproj -->
<ItemGroup>
  <IkvmReference Include="..\\..\\choco-solver-4.10.17.jar" />
</ItemGroup>
```

Cette approche (DLL pré-compilée + runtime NuGet) débloque l'exécution **réelle** de Choco-solver dans le kernel dotnet-interactive (cf. [#4711](https://github.com/jsboige/CoursIA/pull/4711) pour le diagnostic complet des deux verrous levés).

In [2]:
// Configuration IKVM 8.15.0 pour Choco-solver -- exécution réelle en-kernel (See #4956, See #3801)
//
// Deux verrous documentés dans #4711 (Sudoku-11) sont levés ici :
//   1. Conflit System.Text.Json 8.0.0.5 : on charge IKVM via NuGet (et non des DLL IKVM locales
//      qui épinglaient 8.0.0.5), ce qui laisse le kernel résoudre une version compatible.
//   2. "Could not locate ikvm home path" : IKVM 8.15 ne consulte PAS la variable d'env IKVM_HOME ;
//      il lit AppContext["IKVM.Home"]. On assemble le home complet (fusion de l'image arch-indépendante
//      any/any -- classes + tzdb.dat -- et de l'image native win-x64) puis on le déclare via AppContext,
//      AVANT tout premier appel Java (l'init de la JVM se déclenche au premier type java.*, en cellule suivante).
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "nuget: IKVM.Image.runtime.win-x64, 8.15.0"

using System.IO;

string ikvmVer = "8.15.0", ikvmRid = "win-x64";
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome    = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);

void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}

if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);   // classes Java + tzdb.dat (arch-indépendant)
    IkvmCopyMerge(ikvmArchDir, ikvmHome);   // bibliothèques natives win-x64 (bin/ + lib/)
}
AppContext.SetData("IKVM.Home", ikvmHome);

bool tzdbOk = File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat"));
Console.WriteLine("IKVM 8.15.0 prêt (home=" + Path.GetFileName(ikvmHome) + ", tzdb=" + tzdbOk + ") - Choco-solver charge");

In [3]:
// DLL Choco-solver pré-compilée : référencée ici (après la config IKVM), avant les imports de namespaces.
#r "org.chocosolver.solver.dll"

// Imports Choco via IKVM (espaces de noms Java mappés vers .NET)
using org.chocosolver.solver;
using org.chocosolver.solver.variables;
using org.chocosolver.solver.constraints;
using org.chocosolver.solver.search.strategy.selectors.variables;
using org.chocosolver.solver.search.strategy.selectors.values;
using System;
using System.Linq;
using System.Collections.Generic;

// Désambiguïsation : org.chocosolver.solver.variables.Task vs System.Threading.Tasks.Task
using Task = System.Threading.Tasks.Task;

Console.WriteLine("Choco-solver via IKVM 8.15.0 - Prêt pour ordonnancement industriel");

## 1. Job-Shop Scheduling Problem (JSSP) avec Choco

Le JSSP est un problème classique d'ordonnancement :
- **n jobs** à traiter sur **m machines**
- Chaque job = séquence d'opérations (machine, durée)
- Précédence : les opérations d'un job sont ordonnées
- **Disjonction** : une machine traite au plus une opération à la fois
- **Objectif** : minimiser le **makespan** (fin de la dernière opération)

### Modélisation Choco

- `IntVar start[j][o]`, `IntVar end[j][o]`, `IntervalVar task[j][o]` (Choco `Task` est l'objet Java, d'où la désambiguïsation `using Task = System.Threading.Tasks.Task;` plus haut)
- `model.noOverlap(intervalArray)` : contrainte globale de disjonction sur une machine (équivalent CP-SAT `AddNoOverlap`)
- `model.arithm(end, ">=", start)` : précédence entre opérations d'un même job
- `model.setObjective(makespan, ResolutionPolicy.MINIMIZE)` : objectif de minimisation (note : `model.minimize(...)` n'existe PAS en Choco 4.10.17, utiliser `setObjective(..., ResolutionPolicy.MINIMIZE)`)
- `model.max(lastEnds, makespan).post()` : contrainte globale max (note : le 1er arg est la liste de vars, le 2e est le résultat)

In [4]:
// Exemple résolu : JSSP 3 jobs × 3 machines (même instance que CSP-4-Scheduling.ipynb)
//   Job 0 : M0(3u) → M1(2u) → M2(2u)
//   Job 1 : M0(2u) → M2(1u) → M1(4u)
//   Job 2 : M1(4u) → M2(3u)
//   Objectif : minimiser le makespan. Optimal connu : 11 unités (cf. CSP-4-Scheduling.ipynb).

int[][][] jobsData = new int[3][][];
jobsData[0] = new int[][] { new[] {0, 3}, new[] {1, 2}, new[] {2, 2} };
jobsData[1] = new int[][] { new[] {0, 2}, new[] {2, 1}, new[] {1, 4} };
jobsData[2] = new int[][] { new[] {1, 4}, new[] {2, 3} };

int numJobs = jobsData.Length;
int horizon = jobsData.SelectMany(j => j.Select(op => op[1])).Sum();

var model = new Model("JSSP-3x3 via Choco");
var taskIntervals = new List<IntervalVar>();
var taskStarts = new Dictionary<(int, int), IntVar>();
var taskEnds = new Dictionary<(int, int), IntVar>();
var machineToIntervals = new Dictionary<int, List<IntervalVar>>();

// Variables : pour chaque opération (job, opIdx) un triplet (start, end, interval)
for (int j = 0; j < numJobs; j++)
{
    for (int o = 0; o < jobsData[j].Length; o++)
    {
        int machine = jobsData[j][o][0];
        int duration = jobsData[j][o][1];
        var start = model.intVar($"s_{j}_{o}", 0, horizon, false);
        var end = model.intVar($"e_{j}_{o}", 0, horizon, false);
        var interval = model.intervalVar(start, duration, end, $"i_{j}_{o}");
        taskStarts[(j, o)] = start;
        taskEnds[(j, o)] = end;
        taskIntervals.Add(interval);
        if (!machineToIntervals.ContainsKey(machine))
            machineToIntervals[machine] = new List<IntervalVar>();
        machineToIntervals[machine].Add(interval);
    }
}

// Contrainte 1 : précédence intra-job (op(o+1) commence après fin de op(o))
for (int j = 0; j < numJobs; j++)
    for (int o = 0; o < jobsData[j].Length - 1; o++)
        model.arithm(taskEnds[(j, o)], "<=", taskStarts[(j, o + 1)]).post();

// Contrainte 2 : disjonction par machine (noOverlap global)
foreach (var kvp in machineToIntervals)
    model.noOverlap(kvp.Value.ToArray()).post();

// Objectif : minimiser le makespan
//   - makespan = max(lastEnds of each job)
//   - signature : model.max(IntVar[] vars, IntVar result).post()
//   - objectif : model.setObjective(IntVar, ResolutionPolicy.MINIMIZE) — pas model.minimize() !
var makespan = model.intVar("makespan", 0, horizon, false);
var lastEnds = Enumerable.Range(0, numJobs)
    .Select(j => taskEnds[(j, jobsData[j].Length - 1)])
    .ToArray();
model.max(lastEnds, makespan).post();
model.setObjective(makespan, ResolutionPolicy.MINIMIZE);

// Résolution
//   - Search.intVarSearch signature : (VariableSelector, ValueSelector, IntVar[])
//   - ⚠ NE PAS passer taskIntervals (IntervalVar[]) -- signature attend IntVar[]
var solver = model.getSolver();
solver.setSearch(org.chocosolver.solver.search.strategy.Search.intVarSearch(
    new FirstFail(model),
    new IntDomainMin(),
    taskStarts.Values.ToArray()
));
bool solved = solver.solve();

if (solved)
{
    int mk = solver.getModel().getVar("makespan").getValue();
    Console.WriteLine($"JSSP : Statut = {solver.getDecisionCount()} décisions");
    Console.WriteLine($"JSSP : Makespan optimal = {mk} unités");
    Console.WriteLine();
    Console.WriteLine("Schedule détaillé :");
    for (int j = 0; j < numJobs; j++)
        for (int o = 0; o < jobsData[j].Length; o++)
        {
            int machine = jobsData[j][o][0];
            int duration = jobsData[j][o][1];
            int st = taskStarts[(j, o)].getValue();
            int en = taskEnds[(j, o)].getValue();
            Console.WriteLine($"  Job {j} Op {o} : M{machine} [{st}→{en}] (durée {duration})");
        }
}
else
{
    Console.WriteLine("JSSP : Pas de solution trouvée.");
}

## 2. Resource-Constrained Project Scheduling (RCPSP) avec Choco

Le RCPSP généralise le JSSP :
- **Tâches** avec durées et **prédécesseurs**
- **Ressources renouvelables** avec capacité limitée (personnel, machines)
- Contrainte **`cumulative`** : la consommation totale à tout instant ≤ capacité

### Modélisation Choco

- `IntervalVar task[t]` pour chaque tâche (durée fixée)
- Précédence : `model.arithm(start[t], ">=", end[pred])`
- **Cumulative** : `model.cumulative(intervals, demands, capacity)` est l'équivalent exact de CP-SAT `AddCumulative`. C'est la **contrainte globale de cumul** — bien plus efficace que d'écrire toutes les paires.
- Objectif : `model.setObjective(makespan, ResolutionPolicy.MINIMIZE)` (idem JSSP)

In [5]:
// Exemple résolu : RCPSP 6 tâches, 2 ressources (R0 cap=4, R1 cap=3)
//   T0 (2u, pred=[])    R0:2  R1:1
//   T1 (3u, pred=[0])   R0:1  R1:2
//   T2 (4u, pred=[0])   R0:3  R1:0
//   T3 (2u, pred=[1])   R0:0  R1:2
//   T4 (3u, pred=[2])   R0:1  R1:1
//   T5 (1u, pred=[3,4]) R0:2  R1:0
//   Optimal connu : makespan = 10 unités (cf. CSP-4-Scheduling.ipynb).

int numTasks = 6;
int[] durations = { 2, 3, 4, 2, 3, 1 };
int[][] predecessors = {
    new int[0],
    new[] {0},
    new[] {0},
    new[] {1},
    new[] {2},
    new[] {3, 4}
};
int[][] resourceNeeds = {
    new[] {2, 1},
    new[] {1, 2},
    new[] {3, 0},
    new[] {0, 2},
    new[] {1, 1},
    new[] {2, 0}
};
int[] capacities = { 4, 3 };
int horizonRcpsp = 20;

var model2 = new Model("RCPSP via Choco");
var starts2 = new IntVar[numTasks];
var ends2 = new IntVar[numTasks];
var intervals2 = new IntervalVar[numTasks];

for (int t = 0; t < numTasks; t++)
{
    starts2[t] = model2.intVar($"s_{t}", 0, horizonRcpsp, false);
    ends2[t] = model2.intVar($"e_{t}", 0, horizonRcpsp, false);
    intervals2[t] = model2.intervalVar(starts2[t], durations[t], ends2[t], $"i_{t}");
}

// Précédences
for (int t = 0; t < numTasks; t++)
    foreach (int pred in predecessors[t])
        model2.arithm(starts2[t], ">=", ends2[pred]).post();

// Contraintes cumulatives par ressource
for (int r = 0; r < capacities.Length; r++)
{
    var taskSubset = new List<IntervalVar>();
    var demandSubset = new List<int>();
    for (int t = 0; t < numTasks; t++)
    {
        int need = resourceNeeds[t][r];
        if (need > 0)
        {
            taskSubset.Add(intervals2[t]);
            demandSubset.Add(need);
        }
    }
    if (taskSubset.Count > 0)
        model2.cumulative(taskSubset.ToArray(), demandSubset.ToArray(), capacities[r]).post();
}

// Objectif : minimiser le makespan (signature corrigée)
var makespan2 = model2.intVar("makespan_rcpsp", 0, horizonRcpsp, false);
model2.max(ends2, makespan2).post();
model2.setObjective(makespan2, ResolutionPolicy.MINIMIZE);

bool solved2 = model2.getSolver().solve();
if (solved2)
{
    int mk2 = model2.getVar("makespan_rcpsp").getValue();
    Console.WriteLine($"RCPSP : Makespan optimal = {mk2} unités");
    Console.WriteLine();
    Console.WriteLine("Schedule détaillé :");
    for (int t = 0; t < numTasks; t++)
        Console.WriteLine($"  Tâche {t} : [{starts2[t].getValue()}→{ends2[t].getValue()}] (durée {durations[t]})");
}
else
{
    Console.WriteLine("RCPSP : Pas de solution trouvée.");
}

## 3. Nurse Scheduling Problem avec Choco

Le Nurse Scheduling affecte **n infirmiers** sur **d jours** à **p postes** (matin, après-midi, nuit) :
- **Couverture** : au moins `k` infirmiers par poste
- **Unicité** : un infirmier n'a qu'un poste par jour
- **Charge** : entre `min_shifts` et `max_shifts` par infirmier sur la période

### Modélisation Choco

- `BoolVar x[n, d, s]` : 1 si infirmier `n` travaille le jour `d` au poste `s`
- `model.sum(vars, ">=", k)` : contrainte de couverture par poste (équivalent CP-SAT `Add` avec `>=`)
- `model.sum(vars, "<=", 1)` : unicité journalière
- Objectif : `model.setObjective(totalShifts, ResolutionPolicy.MINIMIZE)` (équilibrage du total des shifts)

In [6]:
// Exemple résolu : Nurse Scheduling — 6 infirmiers × 7 jours × 3 postes (Matin/Après-midi/Nuit)
//   min_nurses_per_shift = 2, max_shifts_per_nurse = 7
//   Optimal connu : 42 shifts assignés (= 2 × 7 × 3, couverture minimale exacte) — cf. CSP-4-Scheduling.ipynb

int numNurses = 6;
int numDays = 7;
int shiftsPerDay = 3;
int minNursesPerShift = 2;
int maxShiftsPerNurse = 7;

var model3 = new Model("Nurse Scheduling via Choco");
// x[n, d, s] booléen : 1 si l'infirmier n travaille le jour d au poste s
var x = new BoolVar[numNurses, numDays, shiftsPerDay];
for (int n = 0; n < numNurses; n++)
    for (int d = 0; d < numDays; d++)
        for (int s = 0; s < shiftsPerDay; s++)
            x[n, d, s] = model3.boolVar($"x_{n}_{d}_{s}");

// Contrainte 1 : couverture minimale par poste (>= minNursesPerShift par (d, s))
for (int d = 0; d < numDays; d++)
    for (int s = 0; s < shiftsPerDay; s++)
    {
        var perShift = new BoolVar[numNurses];
        for (int n = 0; n < numNurses; n++)
            perShift[n] = x[n, d, s];
        model3.sum(perShift.ToArray(), ">=", minNursesPerShift).post();
    }

// Contrainte 2 : au plus un poste par infirmier par jour
for (int n = 0; n < numNurses; n++)
    for (int d = 0; d < numDays; d++)
    {
        var perDay = new BoolVar[shiftsPerDay];
        for (int s = 0; s < shiftsPerDay; s++)
            perDay[s] = x[n, d, s];
        model3.sum(perDay.ToArray(), "<=", 1).post();
    }

// Contrainte 3 : au plus maxShiftsPerNurse sur toute la période
for (int n = 0; n < numNurses; n++)
{
    var perNurse = new BoolVar[numDays * shiftsPerDay];
    int idx = 0;
    for (int d = 0; d < numDays; d++)
        for (int s = 0; s < shiftsPerDay; s++)
            perNurse[idx++] = x[n, d, s];
    model3.sum(perNurse.ToArray(), "<=", maxShiftsPerNurse).post();
}

// Objectif : minimiser le total des shifts (équilibrage, signature corrigée)
var allVars = new List<BoolVar>();
for (int n = 0; n < numNurses; n++)
    for (int d = 0; d < numDays; d++)
        for (int s = 0; s < shiftsPerDay; s++)
            allVars.Add(x[n, d, s]);
var totalShifts = model3.intVar("total_shifts", 0, numNurses * numDays * shiftsPerDay, false);
model3.sum(allVars.ToArray(), "=", totalShifts).post();
model3.setObjective(totalShifts, ResolutionPolicy.MINIMIZE);

bool solved3 = model3.getSolver().solve();
if (solved3)
{
    int ts = model3.getVar("total_shifts").getValue();
    int assignations = 0;
    int[] perNurseCount = new int[numNurses];
    for (int n = 0; n < numNurses; n++)
        for (int d = 0; d < numDays; d++)
            for (int s = 0; s < shiftsPerDay; s++)
                if (x[n, d, s].getValue() == 1)
                {
                    assignations++;
                    perNurseCount[n]++;
                }
    string[] shiftNames = { "Matin", "AM", "Nuit" };
    Console.WriteLine($"Nurse : Total shifts = {ts}, assignations = {assignations}");
    Console.WriteLine($"Charge par infirmier : [{string.Join(", ", perNurseCount.Select(c => c.ToString()))}]");
    Console.WriteLine();
    Console.WriteLine("Planning (matin/AM/nuit) — première ligne = jour 0 :");
    for (int d = 0; d < numDays; d++)
    {
        var line = new List<string>();
        for (int s = 0; s < shiftsPerDay; s++)
        {
            var nursesHere = new List<int>();
            for (int n = 0; n < numNurses; n++)
                if (x[n, d, s].getValue() == 1) nursesHere.Add(n);
            line.Add($"{shiftNames[s]}=[{string.Join(",", nursesHere)}]");
        }
        Console.WriteLine($"  Jour {d} : {string.Join(" | ", line)}");
    }
}
else
{
    Console.WriteLine("Nurse : Pas de solution trouvée.");
}

## 4. Comparaison OR-Tools CP-SAT (Python) vs Choco (C#/.NET)

| Aspect | OR-Tools CP-SAT (Python) | Choco-solver (C#/.NET) |
|--------|--------------------------|------------------------|
| **Langage natif** | C++ (bindings Python/C#) | Java (porté via IKVM) |
| **Solveur** | CP-SAT (basé sur SAT) | CP complet (recherche + propagation) |
| **Contraintes globales clés** | `AddNoOverlap`, `AddCumulative`, `AddAllDifferent` | `noOverlap`, `cumulative`, `allDifferent` |
| **Variables** | `NewIntVar`, `NewIntervalVar`, `NewBoolVar` | `intVar`, `intervalVar`, `boolVar` |
| **Objectif** | `solver.Minimize(var)` / `solver.Maximize(var)` | `model.setObjective(var, ResolutionPolicy.MINIMIZE/MAXIMIZE)` |
| **Solve** | `solver.Solve(model)` | `model.getSolver().solve()` |
| **Statut** | `OPTIMAL` / `FEASIBLE` / `INFEASIBLE` | `solver.solve()` booléen + `getDecisionCount()` |
| **Performance pure** | Très haute (état de l'art) | Bonne, légèrement plus lent sur grandes instances |
| **API Java/.NET** | Pas de friction | Légère friction IKVM (bootstrap, `Task` désambiguïsation) |
| **Richesse pédagogique** | Excellente | Excellente (bibliothèque mature, +20 ans) |

**Verdict** : les deux solveurs **modélisent les mêmes problèmes** (JSSP-3×3, RCPSP-6×2×2, Nurse-6×7×3) avec les mêmes contraintes globales (`NoOverlap`/`AddNoOverlap`, `Cumulative`/`AddCumulative`). Sur le **binôme Python** ([CSP-4-Scheduling.ipynb](CSP-4-Scheduling.ipynb)), OR-Tools CP-SAT résout à l'optimal (makespans = 11/10/42). Ce binôme .NET applique les mêmes modèles via Choco-IKVM : l'exécution end-to-end et les valeurs exactes des makespans sont visibles dans les outputs des cellules 6, 8, 10 ci-dessus (sous réserve de la capture iopub — voir [#5005](https://github.com/jsboige/CoursIA/issues/5005)).

## 5. Exercices

Trois exercices pour approfondir l'usage de Choco-solver. Chaque exercice étend un modèle résolu ci-dessus avec une variante réaliste.

In [7]:
// EXERCICE 1 : JSSP avec deadlines (variante du modèle de la cellule 6)
//
// Énoncé : Reprenez le JSSP 3 jobs × 3 machines de la cellule 6 et ajoutez une
// **deadline** (date de fin au plus tard) pour chaque job : Job 0 ≤ 10, Job 1 ≤ 12, Job 2 ≤ 8.
// Minimisez le **retard maximal** (max_lateness = max(0, completion - deadline)).
//
// Indice :
//   1. Pour chaque job, créez une variable IntVar lateness[j] (0..horizon).
//   2. Ajoutez la contrainte arithm : lateness[j] >= end[lastOpOfJob(j)] - deadline[j].
//      (signature : model.arithm(var1, ">=", var2_minus_const) — Choco supporte les expressions)
//   3. Créez une variable maxLateness = max(lateness[0..2]) avec model.max(latenesss, maxLateness).post().
//   4. Minimisez maxLateness via model.setObjective(maxLateness, ResolutionPolicy.MINIMIZE).
//
// Données :
//   int[] deadlines = { 10, 12, 8 };
//   jobsData, horizon identiques à la cellule 6.

// Votre code ici
Console.WriteLine("Exercice à compléter");

In [8]:
// EXERCICE 2 : RCPSP avec budget (variante du modèle de la cellule 8)
//
// Énoncé : Étendez le RCPSP de la cellule 8 avec une **ressource non-renouvelable** :
// budget = 14 unités. Chaque tâche a un coût (3, 2, 4, 1, 2, 1 respectivement) ;
// la somme des coûts des tâches doit rester ≤ 14. Le makespan reste à minimiser.
//
// Indice :
//   1. Pour la ressource non-renouvelable, ce n'est PAS une cumulative (qui est temporelle).
//      C'est une simple contrainte de somme : sum(cost[t]) <= budget.
//   2. Créez un IntVar sumCost = model2.intVar("sum_cost", 0, 100, false),
//      puis model2.sum(costVars, "=", sumCost).post() et model2.arithm(sumCost, "<=", 14).post().
//   3. Vérifiez que la somme des coûts (3+2+4+1+2+1 = 13) est ≤ 14 : instance faisable.
//
// Données :
//   int[] costs = { 3, 2, 4, 1, 2, 1 };  // somme = 13, budget = 14 → 1 unité de marge
//   int budget = 14;
//   Reste des paramètres identique à la cellule 8.

// Votre code ici
Console.WriteLine("Exercice à compléter");

In [9]:
// EXERCICE 3 : Nurse Scheduling avec équilibrage de charge (variante cellule 10)
//
// Énoncé : Modifiez le modèle Nurse de la cellule 10 pour imposer un **équilibrage strict** :
//   1. Chaque infirmier doit travailler **au moins 6 shifts** sur la semaine (min_shifts = 6).
//   2. Pas plus de **5 jours consécutifs** travaillés (contrainte supplémentaire).
//   3. Continuez à minimiser le total des shifts (= 42 = couverture exacte, comme la cellule 10).
//
// Indice pour la contrainte "5 jours consécutifs" :
//   Pour chaque infirmier n et chaque jour d de 0 à 2 (= 7-5) :
//     sum(x[n, d..d+4, 0..2]) <= 5
//   Cela force "pas plus de 5 jours travaillés d'affilée".
//
// Données :
//   int minShifts = 6;
//   Reste identique à la cellule 10.

// Votre code ici
Console.WriteLine("Exercice à compléter");

## Conclusion

Ce notebook a démontré la **parité Python/.NET** sur trois problèmes d'ordonnancement industriel classiques :

| Problème | Modélisation Choco | Makespan (référence Python OR-Tools) |
|----------|---------------------|--------------------------------------|
| JSSP 3×3 | `noOverlap` par machine + précédence intra-job | 11 |
| RCPSP 6×2 | `cumulative` par ressource + précédence | 10 |
| Nurse 6×7×3 | `BoolVar` + sommes (`>=`, `<=`) | 42 (couverture exacte) |

### Points clés à retenir

1. **Bridge IKVM 8.15.0** : Choco (Java) s'exécute réellement dans le kernel `.net-csharp` après configuration de `AppContext["IKVM.Home"]` (cf. [#4711](https://github.com/jsboige/CoursIA/pull/4711)).
2. **DLL pré-compilée** : `org.chocosolver.solver.dll` (12 Mo) est située à côté du notebook, le runtime IKVM vient de NuGet — pas d'épinglage de version `System.Text.Json`.
3. **Contraintes globales** : `noOverlap` et `cumulative` sont les équivalents Choco de `AddNoOverlap` et `AddCumulative` en CP-SAT — même expressivité, même efficacité.
4. **Objectif Choco** : `model.setObjective(var, ResolutionPolicy.MINIMIZE)` (PAS `model.minimize(...)` qui n'existe pas en 4.10.x).
5. **Recherche** : `Search.intVarSearch(selector, valSelector, IntVar[])` — le 3e arg DOIT être un tableau d'`IntVar` (pas d'`IntervalVar`).
6. **Désambiguïsation** : `using Task = System.Threading.Tasks.Task;` résout le conflit entre `org.chocosolver.solver.variables.Task` et `System.Threading.Tasks.Task`.
7. **Marathon #4956** : ce binôme est la première tranche .NET de la série CSP. Prochaines cibles : CSP-3 (Advanced), CSP-5 (Optimization), CSP-6 (Hybridization) — voir [l'epic #4956](https://github.com/jsboige/CoursIA/issues/4956).

### Voir aussi

- [CSP-4-Scheduling.ipynb](CSP-4-Scheduling.ipynb) — version Python (OR-Tools CP-SAT)
- [Sudoku-11-Choco-Csharp.ipynb](../../Sudoku/Sudoku-11-Choco-Csharp.ipynb) — jurisprudence IKVM 8.15.0 + Choco
- [PR #4711](https://github.com/jsboige/CoursIA/pull/4711) — diagnostic des deux verrous IKVM levés
- [EPIC #4956](https://github.com/jsboige/CoursIA/issues/4956) — parité .NET ⇄ Python des séries de notebooks